# JobCompass — Notebook 04: OpenSearch Indexing

## What this does
Creates two production-grade OpenSearch indices and bulk-loads all candidates and jobs.

Each index supports two retrieval modes simultaneously:
- **BM25 lexical search** — custom analyzers for skills and job titles
- **Dense vector ANN search** — Lucene HNSW for approximate nearest-neighbour

NB05 queries both in parallel and fuses results with Reciprocal Rank Fusion.

## Why Lucene engine (not faiss, not nmslib)
OpenSearch 2.x ships three kNN engines. The right choice depends on your use case:

| Engine | Best for | Weakness |
|--------|----------|----------|
| **Lucene** | <10M vectors, real-time updates, production simplicity | Slower build vs faiss on huge datasets |
| nmslib | Legacy, large static datasets | Deprecated in OS 2.12+ |
| faiss | >10M vectors, GPU indexing, research | Requires vectors in JVM heap, brittle ops |

For JobCompass (~2k candidates, ~5k jobs), Lucene is correct:
- Vectors stored on disk with memory-mapped files — no heap pressure
- Real-time indexing works (new candidates appear instantly)
- No kNN circuit breaker tuning required
- Officially recommended by OpenSearch for datasets under 10M vectors

## Index architecture
```
candidates_v1  ←── alias: candidates  (what NB05 queries)
jobs_v1        ←── alias: jobs
```
All queries target aliases. To reindex, create `_v2`, load it, atomically swap the alias.

## Prerequisites — start OpenSearch with Docker
```bash
docker run -d \
  --name opensearch \
  -p 9200:9200 \
  -e discovery.type=single-node \
  -e DISABLE_SECURITY_PLUGIN=true \
  -e OPENSEARCH_JAVA_OPTS="-Xms2g -Xmx2g" \
  opensearchproject/opensearch:2.13.0
```
Wait 30s, verify: `curl http://localhost:9200` → should return cluster JSON.

## Instructions
1. Start OpenSearch container (above)
2. Ensure NB03 complete — `candidates_embedded.jsonl` and `jobs_embedded.jsonl` exist
3. Run all cells top to bottom
4. Rerun safe — existing populated indices are skipped unless `FORCE_REINDEX = True`

In [1]:
# Cell 0 — Install dependencies
import subprocess, sys

packages = [
    "opensearch-py>=2.4.0",
    "orjson",
    "tqdm",
]
for pkg in packages:
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg],
                       capture_output=True, text=True)
    print(f"  {'OK' if r.returncode == 0 else 'FAIL'} {pkg}")
print("Done.")

  OK opensearch-py>=2.4.0
  OK orjson
  OK tqdm
Done.


In [2]:
# Cell 1 — Configuration
import os, warnings
from pathlib import Path
warnings.filterwarnings("ignore")

# ── Input files (from NB03) ────────────────────────────────────────────────────
PROCESSED_DIR    = "../data/processed"
CANDIDATES_JSONL = f"{PROCESSED_DIR}/candidates_embedded.jsonl"
JOBS_JSONL       = f"{PROCESSED_DIR}/jobs_embedded.jsonl"

# ── OpenSearch ─────────────────────────────────────────────────────────────────
OS_HOST          = "localhost"
OS_PORT          = 9200

# ── Index + alias names ────────────────────────────────────────────────────────
# Versioned index names allow zero-downtime reindexing.
# NB05 always queries the alias — never the versioned index directly.
CANDIDATES_INDEX = "candidates_v1"
JOBS_INDEX       = "jobs_v1"
CANDIDATES_ALIAS = "candidates"
JOBS_ALIAS       = "jobs"

# ── Vector config — must match NB03 exactly ───────────────────────────────────
EMBEDDING_DIM    = 1024    # Qwen3-Embedding-0.6B

# ── HNSW parameters (Lucene engine) ───────────────────────────────────────────
# m: graph connectivity. 16 is standard. Higher = better recall, more memory.
# ef_construction: build-time beam width. 100 is standard production value.
HNSW_M               = 16
HNSW_EF_CONSTRUCTION = 100

# ── Bulk indexing ──────────────────────────────────────────────────────────────
BULK_BATCH_SIZE  = 500     # documents per bulk request

# ── Reindex flag ──────────────────────────────────────────────────────────────
# False: skip if index already has documents (safe rerun)
# True : delete and recreate from scratch
FORCE_REINDEX    = False

for p in [CANDIDATES_JSONL, JOBS_JSONL]:
    if not Path(p).exists():
        raise FileNotFoundError(f"Not found: {p} — run NB03 first.")

print("Config ready.")
print(f"  OpenSearch       : {OS_HOST}:{OS_PORT}")
print(f"  Candidates index : {CANDIDATES_INDEX}  alias → {CANDIDATES_ALIAS}")
print(f"  Jobs index       : {JOBS_INDEX}  alias → {JOBS_ALIAS}")
print(f"  Vector dim       : {EMBEDDING_DIM}")
print(f"  HNSW m           : {HNSW_M}  ef_construction : {HNSW_EF_CONSTRUCTION}")
print(f"  Bulk batch       : {BULK_BATCH_SIZE}")
print(f"  Force reindex    : {FORCE_REINDEX}")

Config ready.
  OpenSearch       : localhost:9200
  Candidates index : candidates_v1  alias → candidates
  Jobs index       : jobs_v1  alias → jobs
  Vector dim       : 1024
  HNSW m           : 16  ef_construction : 100
  Bulk batch       : 500
  Force reindex    : False


In [3]:
# Cell 2 — Connect + health check
from opensearchpy import OpenSearch, RequestsHttpConnection

client = OpenSearch(
    hosts            = [{"host": OS_HOST, "port": OS_PORT}],
    http_compress    = True,
    use_ssl          = False,
    verify_certs     = False,
    ssl_show_warn    = False,
    connection_class = RequestsHttpConnection,
    timeout          = 60,
    max_retries      = 3,
    retry_on_timeout = True,
)

try:
    info   = client.info()
    health = client.cluster.health()
    print(f"OpenSearch : connected ✓")
    print(f"  Version  : {info['version']['number']}")
    print(f"  Cluster  : {info['cluster_name']}")
    print(f"  Status   : {health['status']}")
    if health['status'] == 'red':
        raise RuntimeError("Cluster status is RED — check Docker container logs.")
except ConnectionError as e:
    raise RuntimeError(
        f"Cannot connect to OpenSearch at {OS_HOST}:{OS_PORT}.\n"
        f"Start the Docker container first (see notebook header).\n{e}"
    )

OpenSearch : connected ✓
  Version  : 2.13.0
  Cluster  : docker-cluster
  Status   : green


In [4]:
# Cell 3 — Index settings and mappings
#
# ANALYZER DESIGN
# ───────────────
# Standard text fields use OpenSearch's default analyzer (standard tokenizer +
# lowercase filter). This works well for prose (descriptions, summaries).
#
# Skills and job titles need a custom analyzer because:
#   - Skills contain dots and slashes: "Node.js", "C++", "ASP.NET"
#   - Skills are often exact phrases: "machine learning" != "learning machine"
#   - BM25 on a tokenized "Node.js" splits it into ["node", "js"] losing the identity
#
# Solution: two sub-fields per skill field
#   .text    → analyzed with standard tokenizer (fuzzy recall)
#   .keyword → not analyzed, exact match (precision filter)
#
# SHARD / REPLICA SETTINGS
# ─────────────────────────
# Single-node development setup: 1 shard, 0 replicas.
# OpenSearch cannot allocate replicas without additional nodes.
# For a multi-node production cluster, set replicas=1.
#
# KNN SETTINGS
# ─────────────
# knn=true enables the kNN plugin for this index.
# ef_search controls search-time beam width. 100 is a good production default.
# It can be updated at query time via the ef_search parameter for a precision/speed tradeoff.

COMMON_SETTINGS = {
    "number_of_shards"  : 1,
    "number_of_replicas": 0,
    "knn"               : True,
    "knn.algo_param.ef_search": 100,
    "refresh_interval"  : "30s",     # batched refreshes during bulk load; set to "1s" after
}

KNN_FIELD = {
    "type"      : "knn_vector",
    "dimension" : EMBEDDING_DIM,
    "method"    : {
        "name"      : "hnsw",
        "engine"    : "lucene",         # Lucene: disk-backed, real-time updates, no heap pressure
        "space_type": "cosinesimil",    # matches L2-normalised vectors from NB03
        "parameters": {
            "m"              : HNSW_M,
            "ef_construction": HNSW_EF_CONSTRUCTION,
        }
    }
}

# Helper: a text field with both analyzed and keyword sub-fields
def text_and_keyword(boost: float = 1.0) -> dict:
    return {
        "type"    : "text",
        "boost"   : boost,
        "fields"  : {
            "keyword": {"type": "keyword", "ignore_above": 256}
        }
    }


CANDIDATES_BODY = {
    "settings": COMMON_SETTINGS,
    "mappings": {
        "dynamic": "strict",   # reject unknown fields — prevents mapping explosion
        "properties": {
            # ── Identifiers ───────────────────────────────────────────────────
            "candidate_id"          : {"type": "keyword"},
            "source_file"           : {"type": "keyword", "index": False},
            # stored for offline evaluation ONLY — never used in retrieval queries
            "source_category"       : {"type": "keyword", "index": False},
            "indexed_at"            : {"type": "date"},
            "extraction_method"     : {"type": "keyword", "index": False},

            # ── Contact ───────────────────────────────────────────────────────
            "name"                  : {"type": "text"},
            "email"                 : {"type": "keyword"},
            "phone"                 : {"type": "keyword", "index": False},
            "location": {
                "type": "object",
                "properties": {
                    "city"   : {"type": "keyword"},
                    "country": {"type": "keyword"},
                }
            },

            # ── Professional ─────────────────────────────────────────────────
            "current_title"          : text_and_keyword(boost=2.0),
            "seniority"              : {"type": "keyword"},
            "total_years_experience" : {"type": "short"},
            "summary"                : {"type": "text"},

            # ── Skills ────────────────────────────────────────────────────────
            # skills_flat is the primary BM25 field — all skills in one list.
            # boost=3 reflects that skill match is the most important BM25 signal.
            "skills_flat": text_and_keyword(boost=3.0),
            "skills": {
                "type": "object",
                "properties": {
                    "technical" : text_and_keyword(boost=3.0),
                    "tools"     : text_and_keyword(boost=2.0),
                    "soft"      : {"type": "text"},
                    "languages" : {"type": "keyword"},
                }
            },

            # ── Experience ────────────────────────────────────────────────────
            "work_experience": {
                "type": "nested",
                "properties": {
                    "company"        : text_and_keyword(),
                    "title"          : {"type": "text"},
                    "start_date"     : {"type": "keyword"},
                    "end_date"       : {"type": "keyword"},
                    "duration_months": {"type": "short"},
                    "description"    : {"type": "text"},
                }
            },

            # ── Education ─────────────────────────────────────────────────────
            "education": {
                "type": "nested",
                "properties": {
                    "institution"    : text_and_keyword(),
                    "degree"         : {"type": "keyword"},
                    "field"          : {"type": "text"},
                    "graduation_year": {"type": "short"},
                }
            },
            "certifications"   : {"type": "keyword"},
            "languages_spoken" : {"type": "keyword"},

            # ── Full text for BM25 ────────────────────────────────────────────
            "raw_text"         : {"type": "text"},

            # ── Quality metadata ──────────────────────────────────────────────
            "profile_quality_score" : {"type": "short"},
            "is_poor_quality"       : {"type": "boolean"},
            "missing_fields"        : {"type": "keyword"},
            "page_count"            : {"type": "short"},

            # ── Stored only — not indexed ──────────────────────────────────────
            "embedding_text"   : {"type": "keyword", "index": False},

            # ── Dense vector ──────────────────────────────────────────────────
            "description_vector": KNN_FIELD,
        }
    }
}


JOBS_BODY = {
    "settings": COMMON_SETTINGS,
    "mappings": {
        "dynamic": "strict",
        "properties": {
            # ── Identifiers ───────────────────────────────────────────────────
            "job_id"     : {"type": "keyword"},
            "indexed_at" : {"type": "date"},
            "posted_at"  : {"type": "date", "ignore_malformed": True},
            "is_active"  : {"type": "boolean"},

            # ── Job details ───────────────────────────────────────────────────
            "title"            : text_and_keyword(boost=2.0),
            "normalized_title" : text_and_keyword(boost=2.0),
            "role"             : {"type": "text"},
            "company"          : text_and_keyword(),
            "industry"         : {"type": "keyword"},
            "work_type"        : {"type": "keyword"},
            "company_size"     : {"type": "integer"},
            "seniority"        : {"type": "keyword"},

            # ── Location ──────────────────────────────────────────────────────
            "location": {
                "type": "object",
                "properties": {
                    "city"     : {"type": "keyword"},
                    "country"  : {"type": "keyword"},
                    "latitude" : {"type": "float"},
                    "longitude": {"type": "float"},
                }
            },

            # ── Compensation ──────────────────────────────────────────────────
            "salary": {
                "type": "object",
                "properties": {
                    "min"     : {"type": "integer"},
                    "max"     : {"type": "integer"},
                    "currency": {"type": "keyword"},
                    "raw"     : {"type": "keyword", "index": False},
                }
            },
            "experience_required": {
                "type": "object",
                "properties": {
                    "min_years": {"type": "short"},
                    "max_years": {"type": "short"},
                    "raw"      : {"type": "keyword", "index": False},
                }
            },
            "qualifications" : {"type": "text"},
            "preference"     : {"type": "keyword"},

            # ── Skills ────────────────────────────────────────────────────────
            "skills_flat"        : text_and_keyword(boost=3.0),
            "required_skills"    : text_and_keyword(boost=3.0),
            "nice_to_have_skills": {"type": "text"},
            "tech_stack"         : {"type": "keyword"},
            "domain_tags"        : {"type": "keyword"},

            # ── Content ───────────────────────────────────────────────────────
            "responsibilities_summary": {"type": "text"},
            "description"             : {"type": "text"},
            "responsibilities"        : {"type": "text"},
            "benefits"                : {"type": "text"},
            "raw_text"                : {"type": "text"},

            # ── Stored only ───────────────────────────────────────────────────
            "company_profile" : {"type": "object", "enabled": False},
            "embedding_text"  : {"type": "keyword", "index": False},
            "job_portal"      : {"type": "keyword", "index": False},
            "contact_person"  : {"type": "keyword", "index": False},

            # ── Dense vector ──────────────────────────────────────────────────
            "description_vector": KNN_FIELD,
        }
    }
}

print("Mappings defined.")
print(f"  Candidates : {len(CANDIDATES_BODY['mappings']['properties'])} top-level fields")
print(f"  Jobs       : {len(JOBS_BODY['mappings']['properties'])} top-level fields")

Mappings defined.
  Candidates : 26 top-level fields
  Jobs       : 32 top-level fields


In [5]:
# Cell 4 — Index lifecycle management

def create_index_with_alias(
    index_name : str,
    alias_name : str,
    body       : dict,
    force      : bool = False,
) -> bool:
    """
    Creates a versioned index and points an alias at it.
    Returns True if the index was (re)created and needs data loaded.
    Returns False if the index already has data and was skipped.

    The alias is what NB05 queries. This function also handles the
    zero-downtime atomic alias swap used when reindexing.
    """
    exists = client.indices.exists(index=index_name)

    if exists and not force:
        count = client.count(index=index_name)["count"]
        if count > 0:
            print(f"  {index_name:25s}: exists with {count:,} docs — skipping")
            print(f"  Set FORCE_REINDEX=True to recreate from scratch.")
            return False
        client.indices.delete(index=index_name)
        print(f"  {index_name:25s}: existed empty — recreating")
    elif exists and force:
        client.indices.delete(index=index_name)
        print(f"  {index_name:25s}: deleted (FORCE_REINDEX=True)")

    client.indices.create(index=index_name, body=body)
    print(f"  {index_name:25s}: created ✓")

    # Atomic alias swap — safe even if alias doesn't exist yet
    actions = []
    if client.indices.exists_alias(name=alias_name):
        old_indices = list(client.indices.get_alias(name=alias_name).keys())
        for old in old_indices:
            if old != index_name:
                actions.append({"remove": {"index": old, "alias": alias_name}})
    actions.append({"add": {"index": index_name, "alias": alias_name}})
    client.indices.update_aliases(body={"actions": actions})
    print(f"  alias '{alias_name}' → '{index_name}' ✓")
    return True


print("Creating indices...")
print()
cand_needs_load = create_index_with_alias(
    CANDIDATES_INDEX, CANDIDATES_ALIAS, CANDIDATES_BODY, FORCE_REINDEX
)
print()
jobs_needs_load = create_index_with_alias(
    JOBS_INDEX, JOBS_ALIAS, JOBS_BODY, FORCE_REINDEX
)
print()

# Confirm alias routing
for idx, alias in [(CANDIDATES_INDEX, CANDIDATES_ALIAS), (JOBS_INDEX, JOBS_ALIAS)]:
    aliases = list(
        client.indices.get_alias(index=idx)
        .get(idx, {}).get("aliases", {}).keys()
    )
    print(f"  {idx:25s} → {aliases}")

Creating indices...

  candidates_v1            : created ✓
  alias 'candidates' → 'candidates_v1' ✓

  jobs_v1                  : created ✓
  alias 'jobs' → 'jobs_v1' ✓

  candidates_v1             → ['candidates']
  jobs_v1                   → ['jobs']


In [6]:
# Cell 5 — Bulk indexing
import orjson, time
from tqdm.auto import tqdm


def bulk_index(jsonl_path: str, index_name: str, id_field: str, needs_load: bool) -> dict:
    """
    Bulk-loads all records from a JSONL file into an OpenSearch index.

    Approach:
    - Builds bulk request bodies in-memory per batch of BULK_BATCH_SIZE
    - Uses the document's own ID field as the OpenSearch _id
      (idempotent: re-indexing the same doc overwrites cleanly)
    - Parses error responses per-item so a single bad document
      does not fail the entire batch
    - Refreshes the index after completion so documents are
      immediately searchable

    Returns stats dict.
    """
    if not needs_load:
        count = client.count(index=index_name)["count"]
        print(f"  Skipped — {index_name} already has {count:,} documents.")
        return {"total": 0, "indexed": 0, "failed": 0, "elapsed_s": 0}

    records = []
    with open(jsonl_path, "rb") as f:
        for line in f:
            if line.strip():
                records.append(orjson.loads(line))

    if not records:
        raise ValueError(f"No records found in {jsonl_path}")

    print(f"  {len(records):,} records to index into '{index_name}'")

    indexed, failed = 0, 0
    failed_samples  = []
    t_start         = time.time()

    pbar = tqdm(total=len(records), desc=f"  {index_name}", unit="doc")

    for batch_start in range(0, len(records), BULK_BATCH_SIZE):
        batch = records[batch_start : batch_start + BULK_BATCH_SIZE]

        # Bulk format: action line + document line, interleaved
        lines = []
        for doc in batch:
            lines.append(orjson.dumps(
                {"index": {"_index": index_name, "_id": str(doc[id_field])}}
            ))
            # Strip null description_vector if somehow present (defensive)
            clean = {k: v for k, v in doc.items()
                     if not (k == "description_vector" and v is None)}
            lines.append(orjson.dumps(clean))

        body = b"\n".join(lines) + b"\n"

        try:
            # Do NOT pass timeout as a string like "60s" — the Python client
            # expects an int (seconds) or None. The client-level timeout=60
            # set in Cell 2 already applies to every request including bulk.
            resp = client.bulk(body=body)
            if resp.get("errors"):
                for item in resp["items"]:
                    op = item.get("index", {})
                    if op.get("error"):
                        failed += 1
                        if len(failed_samples) < 5:
                            failed_samples.append({
                                "id"    : op.get("_id"),
                                "reason": op["error"].get("reason", "")[:120],
                            })
                    else:
                        indexed += 1
            else:
                indexed += len(batch)
        except Exception as e:
            failed += len(batch)
            print(f"\n  Bulk request error: {e}")

        pbar.update(len(batch))

    pbar.close()

    # Force refresh — makes all newly indexed documents immediately searchable
    client.indices.refresh(index=index_name)

    # Restore refresh interval to real-time after bulk load
    client.indices.put_settings(
        index=index_name,
        body={"index": {"refresh_interval": "1s"}}
    )

    elapsed = time.time() - t_start
    print(f"  Indexed  : {indexed:,}  /  {len(records):,}")
    print(f"  Failed   : {failed}  {'✓' if failed == 0 else '✗'}")
    print(f"  Time     : {elapsed:.1f}s  ({indexed / max(elapsed, 0.001):.0f} doc/s)")

    if failed_samples:
        print(f"  Sample failures:")
        for s in failed_samples:
            print(f"    [{s['id']}] {s['reason']}")

    return {"total": len(records), "indexed": indexed, "failed": failed, "elapsed_s": round(elapsed, 1)}


print("bulk_index() defined.")

bulk_index() defined.


In [7]:
# Cell 6 — Load candidates
print("=" * 55)
print("CANDIDATES")
print("=" * 55)

cand_stats = bulk_index(
    jsonl_path = CANDIDATES_JSONL,
    index_name = CANDIDATES_INDEX,
    id_field   = "candidate_id",
    needs_load = cand_needs_load,
)
print(f"\nDoc count in '{CANDIDATES_INDEX}': {client.count(index=CANDIDATES_INDEX)['count']:,}")

CANDIDATES
  2,887 records to index into 'candidates_v1'


  candidates_v1: 100%|██████████| 2887/2887 [00:14<00:00, 203.64doc/s]


  Indexed  : 2,881  /  2,887
  Failed   : 6  ✗
  Time     : 15.1s  (191 doc/s)
  Sample failures:
    [0751] failed to parse field [education.graduation_year] of type [short] in document with id '0751'. Preview of field's value: 
    [0907] mapping set to strict, dynamic introduction of [Phillipsburg] within [location] is not allowed
    [1280] failed to parse field [education.graduation_year] of type [short] in document with id '1280'. Preview of field's value: 
    [2137] failed to parse field [education.graduation_year] of type [short] in document with id '2137'. Preview of field's value: 
    [2293] mapping set to strict, dynamic introduction of [Phillipsburg] within [location] is not allowed

Doc count in 'candidates_v1': 2,754


In [8]:
# Cell 7 — Load jobs
print("=" * 55)
print("JOBS")
print("=" * 55)

jobs_stats = bulk_index(
    jsonl_path = JOBS_JSONL,
    index_name = JOBS_INDEX,
    id_field   = "job_id",
    needs_load = jobs_needs_load,
)
print(f"\nDoc count in '{JOBS_INDEX}': {client.count(index=JOBS_INDEX)['count']:,}")

JOBS
  5,000 records to index into 'jobs_v1'


  jobs_v1: 100%|██████████| 5000/5000 [00:21<00:00, 230.68doc/s]


  Indexed  : 5,000  /  5,000
  Failed   : 0  ✓
  Time     : 21.9s  (228 doc/s)

Doc count in 'jobs_v1': 5,000


In [9]:
# Cell 8 — Index validation
import orjson

print("Index validation:")
print()

for index_name, alias_name, jsonl_path, id_field in [
    (CANDIDATES_INDEX, CANDIDATES_ALIAS, CANDIDATES_JSONL, "candidate_id"),
    (JOBS_INDEX,       JOBS_ALIAS,       JOBS_JSONL,       "job_id"),
]:
    stats    = client.indices.stats(index=index_name)
    totals   = stats["indices"][index_name]["total"]
    doc_count = totals["docs"]["count"]
    size_mb   = totals["store"]["size_in_bytes"] / 1024**2
    aliases   = list(
        client.indices.get_alias(index=index_name)
        .get(index_name, {}).get("aliases", {}).keys()
    )

    print(f"  {index_name}")
    print(f"    Documents : {doc_count:,}")
    print(f"    Size      : {size_mb:.1f} MB")
    print(f"    Aliases   : {aliases}")

    # ── BM25 test ──────────────────────────────────────────────────────────────
    bm25 = client.search(
        index = alias_name,
        body  = {
            "query" : {"match": {"raw_text": "python developer"}},
            "size"  : 1,
            "_source": [id_field],
        }
    )
    print(f"    BM25 hits : {bm25['hits']['total']['value']:,} for 'python developer'")

    # ── kNN test ───────────────────────────────────────────────────────────────
    # Use the first record's vector as the query vector
    sample = None
    with open(jsonl_path, "rb") as f:
        for line in f:
            if line.strip():
                sample = orjson.loads(line)
                break

    if sample and sample.get("description_vector"):
        knn = client.search(
            index = alias_name,
            body  = {
                "size" : 3,
                "query": {
                    "knn": {
                        "description_vector": {
                            "vector": sample["description_vector"],
                            "k"     : 3,
                        }
                    }
                },
                "_source": [id_field],
            }
        )
        returned = len(knn["hits"]["hits"])
        top_score = knn["hits"]["hits"][0]["_score"] if returned else 0
        print(f"    kNN hits  : {returned} returned, top score={top_score:.4f}")

    print()

Index validation:

  candidates_v1
    Documents : 10,862
    Size      : 64.4 MB
    Aliases   : ['candidates']
    BM25 hits : 569 for 'python developer'
    kNN hits  : 3 returned, top score=1.0000

  jobs_v1
    Documents : 5,000
    Size      : 95.2 MB
    Aliases   : ['jobs']
    BM25 hits : 310 for 'python developer'
    kNN hits  : 3 returned, top score=1.0000



In [10]:
# Cell 9 — BM25 spot-check queries
# Confirms search is returning sensible results before NB05.

def bm25_search(index_alias: str, query: str, fields: list, top_k: int = 5) -> list:
    return client.search(
        index = index_alias,
        body  = {
            "query": {
                "multi_match": {
                    "query"    : query,
                    "fields"   : fields,
                    "type"     : "best_fields",
                    "operator" : "or",
                }
            },
            "size": top_k,
        }
    )["hits"]["hits"]


print("Job search spot-check (BM25):")
for query in ["python developer", "data scientist", "project manager"]:
    hits = bm25_search(
        JOBS_ALIAS, query,
        ["title^3", "normalized_title^3", "required_skills^2", "description"],
        top_k=3
    )
    print(f"\n  '{query}'")
    for h in hits:
        src = h["_source"]
        print(f"    [{h['_score']:6.2f}]  {str(src.get('normalized_title','?'))[:40]:40s}  {str(src.get('company',''))[:25]}")

print()
print("Candidate search spot-check (BM25):")
for query in ["python machine learning", "java spring backend"]:
    hits = bm25_search(
        CANDIDATES_ALIAS, query,
        ["current_title^3", "skills_flat^2", "raw_text"],
        top_k=3
    )
    print(f"\n  '{query}'")
    for h in hits:
        src = h["_source"]
        print(f"    [{h['_score']:6.2f}]  {str(src.get('current_title','?'))[:40]:40s}  skills: {src.get('skills_flat', [])[:3]}")

Job search spot-check (BM25):

  'python developer'
    [ 21.91]  Server Developer                          Glenmark Pharmaceuticals
    [ 21.91]  Backend Web Developer                     AGCO
    [ 21.91]  Data Scientist                            Victrex

  'data scientist'
    [ 55.99]  Data Scientist                            AES
    [ 55.99]  Data Scientist                            Meta Platforms
    [ 55.99]  Data Scientist                            Victrex

  'project manager'
    [ 37.81]  Agile Project Manager                     Hindalco Industries
    [ 37.81]  Construction Project Manager              General Mills
    [ 37.81]  IT Project Manager                        Uber Technologies

Candidate search spot-check (BM25):

  'python machine learning'
    [ 79.82]  Data Scientist                            skills: ['Machine Learning', 'Data Visualization', 'Big Data']
    [ 79.82]  None                                      skills: ['Machine Learning', 'Data Visualizat

In [11]:
# Cell 10 — Summary
print("NB04 complete.")
print()
for idx, alias in [(CANDIDATES_INDEX, CANDIDATES_ALIAS), (JOBS_INDEX, JOBS_ALIAS)]:
    count = client.count(index=idx)["count"]
    print(f"  {idx:25s}  {count:>6,} docs   alias: '{alias}'")
print()
print("Each index has:")
print("  BM25   : title, skills_flat, required_skills, raw_text (with field boosts)")
print("  kNN    : description_vector — Lucene HNSW, cosine similarity, 2048-dim")
print("  Alias  : zero-downtime reindex pattern (NB05 queries alias, never versioned index)")
print()
print("Next: Notebook 05 — Retrieval + Reranking")
print("  bm25_search()     : lexical top-100 per query")
print("  semantic_search() : ANN top-100 via description_vector")
print("  rrf_combine()     : reciprocal rank fusion → merged top-100")
print("  rerank()          : cross-encoder (bge-reranker-v2-m3) → final top-10")
print("  format_results()  : match_score, matched_skills, missing_skills, explanation")

NB04 complete.

  candidates_v1               2,754 docs   alias: 'candidates'
  jobs_v1                     5,000 docs   alias: 'jobs'

Each index has:
  BM25   : title, skills_flat, required_skills, raw_text (with field boosts)
  kNN    : description_vector — Lucene HNSW, cosine similarity, 2048-dim
  Alias  : zero-downtime reindex pattern (NB05 queries alias, never versioned index)

Next: Notebook 05 — Retrieval + Reranking
  bm25_search()     : lexical top-100 per query
  semantic_search() : ANN top-100 via description_vector
  rrf_combine()     : reciprocal rank fusion → merged top-100
  rerank()          : cross-encoder (bge-reranker-v2-m3) → final top-10
  format_results()  : match_score, matched_skills, missing_skills, explanation
